<a href="https://colab.research.google.com/github/kuNalsharma0725/Capstone-Project--AI_ML/blob/main/Part_4.ipynb" target="_parent"><img src="https://colab.research.google.com/assets/colab-badge.svg" alt="Open In Colab"/></a>

In [55]:
import os
import json
import re
import requests
from jsonschema import validate, ValidationError

In [56]:
# 1. API_KEY Setup
try:
  from google.colab import userdata
  OPENROUTER_API_KEY = userdata.get('OPEN_ROUTER_KEY')
  print("API key loaded successfully")
except Exception:
  print("Key not found in colab secrets")

API key loaded successfully


In [57]:
API_URL = "https://openrouter.ai/api/v1/chat/completions"
MODEL   = "deepseek/deepseek-r1:free"

In [58]:
# Set up the LLM API connection:
def call_llm(system_prompt, user_prompt, temperature=0.0, max_tokens=512):

    payload = {
        "model": MODEL,
        "messages": [
            {"role": "system", "content": system_prompt},
            {"role": "user", "content": user_prompt}
        ],
        "temperature": temperature,
        "max_tokens": max_tokens
    }
    headers = {"Authorization": f"Bearer {OPENROUTER_API_KEY}", "Content-Type": "application/json", "HTTP-Referer": "https://your-site-url.com"}

    response = requests.post(API_URL, headers=headers, json=payload)
    if response.status_code != 200:
        print(f"Error: {response.status_code}")
        return None
    return response.json()['choices'][0]['message']['content']

In [59]:
# Calling the function
sys_prom='You are a structured data extractor.'
user_prom="What are the factors affecting a student lifestyle?"
call_llm(sys_prom, user_prom)

Error: 401


In [60]:
#System prompt defines the role
system_prom="""
    You are a structured data extractor.
    Parameters:
        user_msg (str): The user's query or prompt.
        system_msg (str): High-level instructions or persona for the model.
        api_key (str): Bearer token for API authentication.
        model (str): Model identifier to use. Default is 'deepseek/deepseek-r1:free'.
        verbose (bool): If True, also return token usage info. Default False.

    Returns:
        str: The assistant's reply text (verbose=False).
        tuple: (reply_text, usage_dict) when verbose=True.
        None: If the API call fails.
    """

In [61]:
user_promp="What are the factors affecting a student lifestyle?"

In [62]:
#Call the function with temperature=0
call_llm(system_prom, user_promp, temperature=0)


Error: 401


In [63]:
#Call the function with temperature=0.7
call_llm(system_prom, user_promp, temperature=0.7)

Error: 401


In [64]:
# 3. Structured output handling
schema = {
    "type": "object",
    "properties": {
        "student_id": {"type": "string"},
        "age": {"type": "number"},
        "gender": {"type": "boolean"},
        "extracurricular_participation": {"type": "string"},
        "study_hours_per_day": {"type": "number"}
    },
    "required": ["student_id", "age", "gender", "extracurricular_participation", "study_hours_per_day"]
}

In [65]:
# Construct the user prompt from the feature values
def process_data(user_input):
    if has_pii(user_input):
        print(f"Input blocked: PII detected in: {user_input}")
        return None

    system_prompt = "You are a structured data extractor. Output ONLY valid JSON."
    few_shot_prompt = (
        "Extract data from this text: 'What is the factors affecting a students  extracurricular participation "
        "Output: {'gender': 'Female', 'age': 18, 'sleep_hour': 6.5, 'mental_health_rating': 4, 'extracurricular_participation': 'Yes'}\n"
        "Input: " + user_input
    )

    raw_response = call_llm(system_prompt, few_shot_prompt)
    if not raw_response: return None

    try:
        data = json.loads(raw_response.strip())
        validate(instance=data, schema=schema)
        print(f"Success: {data}")
        return data
    except (json.JSONDecodeError, ValidationError) as e:
        print(f"Validation Error: {e}")
        return None

In [66]:
# 4.Guardrails
import re
def has_pii(text):
    email_pattern = r'[a-zA-Z0-9_.+-]+@[a-zA-Z0-9-]+\.[a-zA-Z0-9-.]+'
    phone_pattern = r'\b\d{10}\b|\b\d{3}[-.\s]\d{3}[-.\s]\d{4}\b'
    return bool(re.search(email_pattern, text) or re.search(phone_pattern, text))


In [67]:
# Guardrail check
def process_data(user_input):

    if has_pii(user_input):
        print("Input blocked: PII detected.")
        return None


    raw_response = call_llm(system_prom, user_input)